In [3]:
# THIS NOTEBOOK IS NOT A COMPLETE PROJECT IT IS JUST ME EXPERIMENTING DIFFERENT CONCEPTS 
# RELATED TO THIS PROJECT FOR FULL PROJECT CHECK OTHER FOLDERS AND FILE FOR BETTER LOOK CHECK README AND ABOUT THE PROJECT FILE 


# generate_data
import numpy as np
import pandas as pd
import os

def generate_asteroid_dataset(n_samples: int = 6000, random_seed: int = 42) -> pd.DataFrame:
    rng = np.random.default_rng(random_seed)
    est_diameter_km = rng.lognormal(mean=-2.0, sigma=1.0, size=n_samples)
    est__diameter_km = np.clip(est_diameter_km, 0.001, 15.0)
    relative_velocity_kms = rng.normal(loc=17.0, scale=7.0, size=n_samples)
    relative_velocity_kms = np.clip(relative_velocity_kms, 1.0, 45.0)
    miss_distance_km = rng.lognormal(mean=16.5, sigma=1.3, size=n_samples)
    miss_distance_km = np.clip(miss_distance_km, 50_000, 9_000_000)
    absolute_magnitude = 23 - 5 * np.log10(est_diameter_km + 0.01) + rng.normal(0, 0.5, n_samples)
    orbit_uncertainty = rng.integers(0, 10, size=n_samples)
    inclination_deg = rng.uniform(0, 35, size=n_samples)
    def normalize(x):
        return (x - x.min()) / (x.max() - x.min() + 1e-9)
    size_score = normalize(est_diameter_km)
    speed_score = normalize(relative_velocity_kms)
    closeness_score = 1 - normalize(miss_distance_km)
    uncertainty_score = normalize(orbit_uncertainty)
    risk_score = (
        0.40 * size_score +
        0.20 * speed_score +
        0.30 * closeness_score +
        0.10 * uncertainty_score
    )
    risk_score_noisy = risk_score + rng.normal(0, 0.07, n_samples)
    cutoff = np.percentile(risk_score_noisy, 94)
    is_hazardous = (risk_score_noisy >= cutoff).astype(int)
    df = pd.DataFrame({
        "est_diameter_km": est_diameter_km,
        "relative_velocity_kms": relative_velocity_kms,
        "miss_distance_km": miss_distance_km,
        "absolute_magnitude": absolute_magnitude,
        "orbit_uncertainty": orbit_uncertainty,
        "inclination_deg": inclination_deg,
        "is_hazardous": is_hazardous,
    })
    return df
if __name__ == "__main__":

    dataset = generate_asteroid_dataset()
    output_dir = "data"
    os.makedirs(output_dir, exist_ok=True)

    output_path = os.path.join(output_dir, "asteroids.csv")
    dataset.to_csv(output_path, index=False)
    hazardous_count = dataset["is_hazardous"].sum()
    total_count = len(dataset)
    print(f"Saved {total_count} simulated asteroids to {output_path}")
    print(f"Hazardous: {hazardous_count} ({hazardous_count / total_count:.1%})")
    print(f"Safe:      {total_count - hazardous_count} ({1 - hazardous_count / total_count:.1%})")


Saved 6000 simulated asteroids to data\asteroids.csv
Hazardous: 360 (6.0%)
Safe:      5640 (94.0%)


In [ ]:
# train_model

import joblib
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import trian_test_split
from sklearn.preprocessing import StandardScaler

# sINCE WE CANNOT IMPORT WITHOUT HAVING A FILE NAMED UTILS INSIDE THE FOLDER
# I AM GOING WITH THE STRCUTURE - TRAINED THE MODEL FROM .PY FILE
from utils import DATASET_PATH, MODEL_PATH, SCALER_PATH, ensure_dirs, section
FEATURE_COLUMNS = [
    "est_diameter_km",
    "relative_velocity_kms",
    "miss_distance_km",
    "absolute_magnitude",
    "orbit_uncertainty",
    "inclination_deg",
]
LABEL_COLUMN = "is_hazardous"

def load_dataset() -> pd.DataFrame:
    return pd.read_csv(DATASET_PATH)

def train():
    ensure_dirs()
    df = load_dataset()
    print(f"Loaded {len(df)} rows from {DATASET_PATH}")

    X = df[FEATURE_COLUMNS]
    y = df[LABEL_COLUMN]

    
     X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=42, stratify=y
    )
    print(f"Training rows: {len(X_train)} | Test rows: {len(X_test)}")
    print(f"Hazardous in training set: {y_train.sum()} ({y_train.mean():.1%})")
    print(f"Hazardous in test set:     {y_test.sum()} ({y_test.mean():.1%})")

    section("STEP 3: Scaling features")
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    section("STEP 4: Training the RandomForestClassifier")
    model = RandomForestClassifier(
        n_estimators=200,        # number of decision trees in the forest
        max_depth=8,              # cap tree depth to avoid overfitting on this small dataset
        class_weight="balanced",  # pay extra attention to the rare hazardous class 
        random_state=42,          # reproducibility
        n_jobs=-1,                 # use all available CPU cores to train faster
    )
    model.fit(X_train_scaled, y_train)
    print("Model training complete.")

     section("STEP 5: Saving model + scaler + test set")
    joblib.dump(model, MODEL_PATH)
    joblib.dump(scaler, SCALER_PATH)

    
    import numpy as np
    np.save(MODEL_PATH.replace("hazard_model.joblib", "X_test.npy"), X_test_scaled)
    np.save(MODEL_PATH.replace("hazard_model.joblib", "y_test.npy"), y_test.values)

    print(f"Saved model to:  {MODEL_PATH}")
    print(f"Saved scaler to: {SCALER_PATH}")


if __name__ == "__main__":
    train()
      

In [ ]:
# evaluate the trained model
import os
import joblib
import numpy as np
import matplotlib
matplotlib.use("Arg")
import matplotlib.pyplot as plt
from sklearn.metrices import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    precision_recall_curve,
    roc_curve,
    roc_auc_score,
)

from utils import MODEL_PATH, SCALER_PATH, OUTPUTS_DIR, section
def load_test_artifacts():
    model = joblib.load(MODEL_PATH)
    X_test = np.load(MODEL_PATH.replace("hazard_model.joblib", "X_test.npy"))
    y_test = np.load(MODEL_PATH.replace("hazard_model.joblib", "y_test.npy"))
    return model, X_test, y_test
def print_confusion_matrix(y_true, y_pred, threshold_label="default (0.5)"):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    print(f"\nConfusion matrix at threshold = {threshold_label}")
    print("                    Predicted SAFE   Predicted HAZARDOUS")
    print(f"  Actually SAFE         TN={tn:<5}         FP={fp:<5}   <- false alarms")
    print(f"  Actually HAZARDOUS    FN={fn:<5}         TP={tp:<5}   <- FN = MISSED THREATS (the dangerous mistake)")
    return tn, fp, fn, tp
def print_metrics(y_true, y_pred):
    """Calculate and print accuracy, precision, recall, specificity, F1."""
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    f1 = f1_score(y_true, y_pred, zero_division=0)

    print(f"  Accuracy:    {accuracy:.3f}  (fraction of ALL predictions correct -- can be misleading, see below)")
    print(f"  Precision:   {precision:.3f}  (of what we flagged hazardous, fraction that really was)")
    print(f"  Recall:      {recall:.3f}  (of all REAL hazards, fraction we caught)  <-- the priority metric")
    print(f"  Specificity: {specificity:.3f}  (of all safe objects, fraction correctly left alone)")
    print(f"  F1 score:    {f1:.3f}  (balanced precision/recall summary)")

    return {
        "accuracy": accuracy, "precision": precision,
        "recall": recall, "specificity": specificity, "f1": f1,
    }

def demonstrate_accuracy_paradox(y_true):
    """
    Show, very concretely, why accuracy alone is a trap on imbalanced data.
    We simulate the laziest possible model: one that predicts "safe" (0)
    for literally every single object, no matter what.
    """
    section("THE ACCURACY PARADOX (a 'do-nothing' model)")
    lazy_predictions = np.zeros_like(y_true) 
    lazy_accuracy = accuracy_score(y_true, lazy_predictions)
    lazy_recall = recall_score(y_true, lazy_predictions, zero_division=0)

    print("A model that predicts 'SAFE' for every single asteroid, with zero intelligence:")
    print(f"  Accuracy: {lazy_accuracy:.1%}   <- looks great on paper!")
    print(f"  Recall:   {lazy_recall:.1%}   <- but it catches ZERO real hazards. Useless in practice.")
    print("This is why we look at the confusion matrix and recall, not just accuracy.")
    

